# Training notebook - neureal-rs-decoder

Trains `PositionPredictor` on a configurable Gilbert-Elliott channel preset. Deisgned to run on Kaggle (T4) but works locally on CPU as well.

Outputs (in `weights/<run_id>/`):
- `model.pth` - final model weights
- `meta.yaml` - training context (config, git, environment, final losses)
- `history.csv` - train/val loss per epoch
<!-- - `checkpoint_epoch_*.pth` - weights checkpoints (TODO) -->

## Workflow
1. Set `BRANCH` and `EXPERIMENT_PARAMS` below.
2. Run all cells.
3. On Kaggle: `File -> Save Version` to persist `weights/` as an output dataset.
4. Locally: copy `weights/<run_id>/model.pth` imto the repo and commit.

## Setup

In [ ]:
# Override config values here without editing default.yaml
# None keeps the config default

REPO_URL = "https://github.com/kayumov-24B81/neural-rs-decoder.git"
BRANCH = "main"

EXPERIMENT_PARAMS = {
    "seed": 42,
    "channel_preset": "moderate", # light/moderate/heavy
    "epochs": None,
    "train_size": None,
    "val_size": None,
}

In [ ]:
# Detect environment and choose working dir
import os
from pathlib import Path

IN_KAGGLE = Path("/kaggle/working").exists()
IN_COLAB = "COLAB_GPU" in os.environ

if IN_KAGGLE:
    os.chdir("/kaggle/working")
    print(f"Kaggle environment. Working dir: {os.getcwd()}")
elif IN_COLAB:
    os.chdir("/content")
    print(f"Colab environment. Working dir: {os.getcwd()}")
else:
    print(f"Local environment. Working dir: {os.getcwd()}")

In [ ]:
# Clone repo (skip if is running locally)

repo_dir = Path("neural-rs-decoder")

if not repo_dir.exists() and (IN_KAGGLE or IN_COLAB):
    !git clone -b {BRANCH} {REPO_URL}
if repo_dir.exists():
    os.chdir(repo_dir)
print(f"Working dir: {os.getcwd()}")

In [ ]:
%pip install -e . --quiet

## Configuration

In [ ]:
import subprocess
import sys
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import yaml

from src.channel import make_ge_channel
from src.dataset import RSPositionDataset
from src.model import FocalLoss, PositionPredictor
from src.train import train_model
from src.utils import load_config, save_model, set_seed
from src.runtime import env_info, git_info

In [ ]:
cfg = load_config("configs/default.yaml")

# Apply overrides
if EXPERIMENT_PARAMS["seed"] is not None:
    cfg["seed"] = EXPERIMENT_PARAMS["seed"]
if EXPERIMENT_PARAMS["channel_preset"] is not None:
    cfg["channel"]["preset"] = EXPERIMENT_PARAMS["channel_preset"]
if EXPERIMENT_PARAMS["epochs"] is not None:
    cfg["training"]["epochs"] = EXPERIMENT_PARAMS["epochs"]
if EXPERIMENT_PARAMS["train_size"] is not None:
    cfg["training"]["train_size"] = EXPERIMENT_PARAMS["train_size"]
if EXPERIMENT_PARAMS["val_size"] is not None:
    cfg["training"]["val_size"] = EXPERIMENT_PARAMS["val_size"]


device = "cuda" if torch.cuda.is_available() else "cpu"
if cfg.get("device", "auto") != "auto":
    device = cfg["device"]

print("Configuration:")
print(f"  seed:         {cfg['seed']}")
print(f"  channel:      {cfg['channel']['preset']}")
print(f"  train_size:   {cfg['training']['train_size']}")
print(f"  val_size:     {cfg['training']['val_size']}")
print(f"  epochs:       {cfg['training']['epochs']}")
print(f"  batch_size:   {cfg['training']['batch_size']}")
print(f"  lr:           {cfg['training']['lr']}")
print(f"  device:       {device}")

In [ ]:
set_seed(cfg["seed"])

# run_id mirrors the convention used by benchmark.py
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_id = f"{timestamp}_{cfg['channel']['preset']}_seed{cfg['seed']}"
run_dir = Path("weights") / run_id
run_dir.mkdir(parents=True, exist_ok=True)
print(f"Run dir: {run_dir}")

## Data

In [ ]:
channel_fn = make_ge_channel(cfg["channel"]["preset"])

train_ds = RSPositionDataset(
    size=cfg["training"]["train_size"],
    channel_fn=channel_fn,
    nsym=cfg["code"]["nsym"],
    msg_len=cfg["code"]["k"],
)
val_ds = RSPositionDataset(
    size=cfg["training"]["val_size"],
    channel_fn=channel_fn,
    nsym=cfg["code"]["nsym"],
    msg_len=cfg["code"]["k"],
)
print(f"Train: {len(train_ds)} blocks, Val: {len(val_ds)} blocks")

# Sanity check: distribution of error counts per block in train set
err_counts = train_ds.positions.sum(axis=1)
print(f"\nErrors per block (train): mean={err_counts.mean():.1f}, "
      f"median={np.median(err_counts):.0f}, "
      f"p95={np.percentile(err_counts, 95):.0f}")

## Training

In [ ]:
model = PositionPredictor(
    input_size=cfg["model"]["input_size"],
    hidden_size=cfg["model"]["hidden_size"],
    dropout=cfg["model"]["dropout"],
)

loss_cfg = cfg["training"]["loss"]
if loss_cfg["type"] == "focal":
    criterion = FocalLoss(
        alpha=loss_cfg["focal_alpha"],
        gamma=loss_cfg["focal_gamma"],
    )
elif loss_cfg["type"] == "bce":
    criterion = torch.nn.BCEWithLogitsLoss()
else:
    raise ValueError(f"Unknown loss type: {loss_cfg['type']}")

print(f"Loss: {type(criterion).__name__}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
history = train_model(
    model=model,
    train_dataset=train_ds,
    val_dataset=val_ds,
    criterion=criterion,
    epochs=cfg["training"]["epochs"],
    batch_size=cfg["training"]["batch_size"],
    lr=cfg["training"]["lr"],
    device=device,
    verbose=True,
    log_every=cfg["training"]["log_every"],
)

## Save artifacts

In [ ]:
save_model(model, run_dir / "model.pth")

history_df = pd.DataFrame({
    "epoch": range(1, len(history["train_loss"]) + 1),
    "train_loss": history["train_loss"],
    "val_loss": history["val_loss"],
})

history_df.to_csv(run_dir / "history.csv", index=False)

meta = {
    "run_id": run_id,
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "git": git_info(),
    "environment": env_info(device),
    "config": cfg,
    "results": {
        "final_train_loss": float(history["train_loss"][-1]),
        "final_val_loss": float(history["val_loss"][-1]),
        "min_val_loss": float(min(history["val_loss"])),
        "min_val_loss_epoch": int(np.argmin(history["val_loss"])) + 1,
    },
}

with open(run_dir / "meta.yaml", "w") as f:
    yaml.safe_dump(meta, f, sort_keys=False, default_flow_style=False)

print(f"Saved: {run_dir}/model.pth")
print(f"Saved: {run_dir}/history.csv")
print(f"Saved: {run_dir}/meta.yaml")
print(f"\nFinal train loss: {history['train_loss'][-1]:.4f}")
print(f"Final val loss:   {history['val_loss'][-1]:.4f}")
print(f"Min val loss:     {min(history['val_loss']):.4f} "
      f"(epoch {np.argmin(history['val_loss']) + 1})")


## Quick loss visualisation

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history_df["epoch"], history_df["train_loss"], label="train")
ax.plot(history_df["epoch"], history_df["val_loss"], label="val")
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.set_title(f"Loss curves — {run_id}")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(run_dir / "loss_curves.png", dpi=120)
plt.show()